# LFM2.5-1.2B-Instruct-Uncensored -> Hindi + Hinglish (Kaggle T4, 30-40 min/session)Trains in **chained sessions** so a Kaggle VM reset never costs more than 5 minutes ofwork: each session trains on the *previous* session's pushed output.| session | data file | trains on ||---|---|---|| 1 | `train_s1_hinglish.jsonl` | the original uncensored base || 2 | `train_s2_hindi.jsonl` | session 1's output || 3 | `train_s3_english.jsonl` | session 2's output |Set `SESSION = 1` below, run, then `SESSION = 2`, etc.**Hindi is Devanagari at 1.45 tok/char** (byte-level BPE, no dedicated Devanagari tokens)vs 0.46 for Hinglish. That's why `MAX_SEQ_LENGTH` is tighter here - it is not a bug.**Never finetune a GGUF.** We train the safetensors base and convert back at the end.

In [ ]:
# ---------------------------------------------------------------- run configSESSION   = 1                      # 1, 2 or 3DATA_FILE = {1: "train_s1_hinglish.jsonl",             2: "train_s2_hindi.jsonl",             3: "train_s3_english.jsonl"}[SESSION]HF_USER   = "YOUR_HF_USERNAME"                  # edit meOUT_REPO  = f"{HF_USER}/lfm25-1.2b-bilingual-s{SESSION}"# Session 1 starts from the uncensored base; sessions 2+ continue from the previous push.BASE_REPO = ("zaakirio/LFM2.5-1.2B-Instruct-Uncensored" if SESSION == 1             else f"{HF_USER}/lfm25-1.2b-bilingual-s{SESSION-1}")CHECKPOINT_EVERY_S = 300            # 5 minutesMAX_STEPS         = {1: 690, 2: 750, 3: 620}[SESSION]print(f"session {SESSION}: {DATA_FILE}\n  base  {BASE_REPO}\n  out   {OUT_REPO}")

## 0. Hugging Face login - REQUIRED`push_to_hub` is the whole point: it is the only thing that survives a Kaggle VM reset.**Without a token, every checkpoint push fails and you lose the run.**1. huggingface.co -> Sign up2. Top-right avatar -> **Settings -> Access Tokens -> Create new token -> `write`**3. Paste it below (or set it as a Kaggle secret named `HF_TOKEN` for a private repo)

### Where the model goesThree repos are created automatically on first push - you don't create them by hand:```https://huggingface.co/<you>/lfm25-1.2b-bilingual-s1     <- session 1 outputhttps://huggingface.co/<you>/lfm25-1.2b-bilingual-s2     <- session 2 (loads s1)https://huggingface.co/<you>/lfm25-1.2b-bilingual-s3     <- session 3 (loads s2)```Each holds the merged model, the `*Q4_K_M.gguf`, and a `ckpt-*` adapter pushed every5 min. Repos are **public by default** - the base model is public, so nothing private isleaked, but check the base repo's licence if you plan to redistribute.Back on the server, pull the final GGUF:```hf download <you>/lfm25-1.2b-bilingual-s3 --include "*Q4_K_M*" --local-dir ~/models/@models```**Set `HF_USER` in the run-config cell above first** - it is the only thing you must edit.

In [ ]:
from huggingface_hub import loginimport osTOKEN = os.environ.get("HF_TOKEN") or "hf_xxxxxxxxxxxxxxxxxxxx"   # or paste it herelogin(token=TOKEN, add_to_git_credential=False)whoami()

In [ ]:
!pip install -q unsloth!pip install -q transformers==4.57.3!pip install -q --no-deps trl==0.22.2!pip install -q datasets==4.3.0 accelerate peft bitsandbytes hf_transfer

## Crash-proof checkpointing**A local save on Kaggle is not a backup** - `/kaggle/working` is destroyed when the VMresets, taking every `save_steps` checkpoint with it. The only thing that survives is aHub push, so this callback pushes the LoRA adapter (small, ~60 MB) on a wall-clock timerrather than a step count. If the session dies at minute 38 you resume from minute 35.

In [ ]:
import os, time, json, globfrom transformers import TrainerCallbackclass TimeCheckpoint(TrainerCallback):    """Push the adapter to the Hub every `every_s` seconds of wall clock."""    def __init__(self, model, repo, every_s=300):        self.model, self.repo, self.every = model, repo, every_s        self.last, self.n = time.time(), 0    def on_step_end(self, args, state, control, **kw):        if time.time() - self.last >= self.every:            self.n += 1            tag = f"ckpt-s{state.global_step//100*100:05d}"            self.model.save_pretrained(f"hf-out/{tag}")            self.model.push_to_hub(self.repo, save_name=tag)            print(f"[{time.strftime('%H:%M:%S')}] pushed {tag} "                  f"({state.global_step} steps, ~{self.model.config.hidden_size})", flush=True)            self.last = time.time()        return controlprint("TimeCheckpoint ready")

## 1. DataUpload the session's jsonl to Kaggle as a Dataset first (Add Data > Upload).

In [ ]:
import torchCANDIDATES = [f"/kaggle/input/hinglish-sft/{DATA_FILE}"] + glob.glob(f"/kaggle/input/*/{DATA_FILE}")DATA_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)if DATA_PATH is None:    raise FileNotFoundError(f"{DATA_FILE} not found under /kaggle/input - upload it as a Dataset")rows = [json.loads(l) for l in open(DATA_PATH, encoding="utf-8")]srcs = {s: sum(1 for r in rows if r["src"] == s) for s in {r["src"] for r in rows}}print(len(rows), "rows |", srcs)dev = sum(1 for r in rows if any("\u0900" <= c <= "\u097F" for c in r["instruction"] + r["response"]))print(f"{dev} rows contain Devanagari")

In [ ]:
MAX_SEQ_LENGTH = 1024      # p90 fits; Hindi rows are ~3x tokens/char so watch the tailDTYPE = torch.float16      # T4 is sm_75, no bf16os.environ["UNSLOTH_RETURN_LOGITS"] = "1"
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"   # -30% VRAM (from the official GRPO notebook)print("GPU:", torch.cuda.get_device_name(0))

## 2. Model + LoRALFM target modules: `q,k,v,out,in_proj,w1,w2,w3` (`in_proj`/`w1,w2,w3` are its MLP).`embed_tokens`/`lm_head` are **not** trained - the tokenizer is unchanged, so there is nonew vocabulary to learn; Devanagari rides on the existing byte-level tokens.

In [ ]:
from unsloth import FastLanguageModel# Guard BEFORE attaching: named_modules() returns dotted paths like# "model.layers.0.self_attn.q_proj", so a bare "q_proj" is never an exact member.# A wrong target name attaches LoRA to nothing and trains happily on zero parameters.TARGET_MODULES = ["q_proj","k_proj","v_proj","out_proj","in_proj","w1","w2","w3"]_names = [n for n, _ in model.named_modules()]_missing = [m for m in TARGET_MODULES            if not any(n == m or n.endswith("." + m) for n in _names)]if _missing:    raise RuntimeError(f"Target modules not present in this model: {_missing}")print("target modules verified:", TARGET_MODULES)model, tokenizer = FastLanguageModel.from_pretrained(    model_name = BASE_REPO, max_seq_length = MAX_SEQ_LENGTH,    dtype = DTYPE, load_in_4bit = False, max_lora_rank = 32,)model = FastLanguageModel.get_peft_model(    model, r = 32, lora_alpha = 32, lora_dropout = 0, bias = "none",    target_modules = TARGET_MODULES,    use_gradient_checkpointing = "unsloth", random_state = 3407, use_rslora = True,)n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)n_all   = sum(p.numel() for p in model.parameters())print(f"trainable {n_train/1e6:.1f}M / {n_all/1e9:.2f}B ({100*n_train/n_all:.2f}%)")if n_train == 0:    raise RuntimeError("LoRA attached to zero parameters - check TARGET_MODULES")

## 3. Chat template`apply_chat_template` **already adds `<|startoftext|>`** and terminates with`<|im_end|>` - do not append an EOS yourself, and do not strip BOS twice.`train_on_responses_only` masks the user turn, so loss lands on the reply only.

In [ ]:
def to_text(batch):    convs = [[{"role":"user","content":i},{"role":"assistant","content":r}]             for i, r in zip(batch["instruction"], batch["response"])]    return {"text": [tokenizer.apply_chat_template(c, tokenize=False,                    add_generation_prompt=False) for c in convs]}from datasets import Datasetds = Dataset.from_list(rows).map(to_text, batched=True, remove_columns=list(rows[0].keys()))split = ds.train_test_split(test_size=0.02, seed=3407)print("train", len(split["train"]), "eval", len(split["test"]))print(repr(split["train"][0]["text"][:180]))

In [ ]:
from unsloth import UnslothTrainer, UnslothTrainingArgumentsargs = UnslothTrainingArguments(    per_device_train_batch_size = 8,    gradient_accumulation_steps = 1,    max_steps = MAX_STEPS,    warmup_steps = 30,    learning_rate = 2e-4,               # lower than 2e-4-per-run for chained sessions    lr_scheduler_type = "linear",    optim = "adamw_8bit",    weight_decay = 0.01,    eval_strategy = "steps", eval_steps = 150,    seed = 3407,    output_dir = "hf-out",    save_strategy = "no",               # the TimeCheckpoint callback owns saving    report_to = [],)trainer = UnslothTrainer(    model = model, tokenizer = tokenizer,    train_dataset = split["train"], eval_dataset = split["test"],    dataset_text_field = "text", max_seq_length = MAX_SEQ_LENGTH, args = args,)trainer.add_callback(TimeCheckpoint(model, OUT_REPO, every_s=CHECKPOINT_EVERY_S))print("trainer ready - checkpoints push every", CHECKPOINT_EVERY_S, "s")

In [ ]:
stats = trainer.train()print(stats.metrics)

## 4. Merge -> GGUF -> Hub`save_pretrained_gguf` runs llama.cpp's converter in-notebook. Push the final merged modelso the next session - and your server - can pull it.Even if this cell never runs, the `ckpt-*` adapters pushed every 5 min are on the Hub.

In [ ]:
model.save_pretrained_gguf("hf-out", tokenizer, quantization_method = "q4_k_m")model.push_to_hub_gguf(OUT_REPO, tokenizer, quantization_method = "q4_k_m")model.push_to_hub(OUT_REPO, tokenizer)      # session 2/3 loads thisprint("done ->", f"https://huggingface.co/{OUT_REPO}")

## 4b. Optional: iMatrix-quantized GGUF (higher quality, same size)`save_pretrained_gguf("q4_k_m")` is a **plain** quantization - it picks outliers bytensor statistics alone. iMatrix first runs the model over a calibration corpus to learnwhich *activations* actually matter, then quantizes with that importance weighting. Same`q4_k_m` bit depth, same file size, measurably less damage.This matters here specifically: the whole reason for finetuning is that the 1.2B scoresonly 5/8 on the accuracy suite, so we cannot afford to lose anything to quantization.Cost: ~5-10 min of extra session time. **Skip it if you are near the 30-minute mark** -the plain GGUF from the previous cell is a complete, working model. Pattern borrowed from`coachvitorcalvi/ava-lab-dyagnosys-finetune-v1`, which exports both and prefers imatrix.

In [ ]:
import glob, subprocessUSE_IMATRIX = True          # set False if you are short on time - plain q4_k_m is fineif not USE_IMATRIX:    print("skipped - use the q4_k_m from the previous cell")else:    # 1. calibration corpus from the actual training data (already formatted)    calib = "hf-out/calib.txt"    with open(calib, "w", encoding="utf-8") as f:        for t in split["train"]["text"][:2000]:            f.write(t + "\n\n")    print("calibration corpus:", os.path.getsize(calib), "bytes")    # 2. f16 GGUF is the input to both imatrix and quantize    model.save_pretrained_gguf("hf-out-f16", tokenizer, quantization_method = "f16")    f16 = sorted(glob.glob("hf-out-f16/**/*.gguf", recursive=True))    assert f16, "no f16 GGUF produced"    f16 = f16[0]    print("f16:", f16)    # 3. Unsloth vendors a llama.cpp build; find the two binaries in it    def find_bin(name):        for root in ("/root/.unsloth/llama.cpp", "/kaggle/working/llama.cpp"):            for q in glob.glob(f"{root}/**/{name}", recursive=True):                if os.access(q, os.X_OK):                    return q        return None    imat, quant = find_bin("llama-imatrix"), find_bin("llama-quantize")    if not (imat and quant):        print("vendored build lacks the quantizers - building llama.cpp (adds ~4 min)")        d = "/kaggle/working/llama.cpp"        if not os.path.isdir(d):            subprocess.run(["git","clone","--depth","1","https://github.com/ggerganov/llama.cpp",d], check=True)        subprocess.run(["cmake","-B",f"{d}/build","-S",d,"-DLLAMA_CURL=OFF"], check=True)        subprocess.run(["cmake","--build",f"{d}/build","--config","Release","-j","-t",                        "llama-imatrix","llama-quantize"], check=True)        imat, quant = f"{d}/build/bin/llama-imatrix", f"{d}/build/bin/llama-quantize"    # 4. learn the importance matrix, then quantize with it    subprocess.run([imat,"-m",f16,"-f",calib,"-o","hf-out/imatrix.dat","--chunks","50"], check=True)    subprocess.run([quant,"--imatrix","hf-out/imatrix.dat",f16,"hf-out/gguf-imatrix","q4_k_m"], check=True)    print("iMatrix q4_k_m:", glob.glob("hf-out/gguf-imatrix*"))    # push whichever one you keep - the imatrix one if it exists    model.push_to_hub_gguf(OUT_REPO, tokenizer, quantization_method="q4_k_m")    print("done - imatrix GGUF is the better artifact, same size")

## 5. Recovering after a resetThe session died but you have checkpoints. Skip straight to the merge with the newest one:```pip install -q unslothmodel, tok = FastLanguageModel.from_pretrained(    model_name="YOUR_HF_USERNAME/lfm25-1.2b-bilingual-s1",    adapter_path="ckpt-s00600", ...)      # newest ckpt-* from the Hub```Then run the merge cell. You lose at most `CHECKPOINT_EVERY_S` of progress.